# Main Figure v2 — Standalone Panels for Illustrator Assembly

Each panel is saved as an **independent figure** (PDF + PNG) with an
accompanying **source-data TSV**. Assemble in Illustrator.

| Panel | File prefix | Content |
|-------|-------------|---------|
| A1 | `panel_A1_strip_box` | Gene agreement — strip + boxplot |
| A2 | `panel_A2_horiz_bar` | Gene agreement — horizontal bar + strip |
| A3 | `panel_A3_violin` | Gene agreement — violin |
| B1 | `panel_B1_intron_chain_8class` | Transcript concordance Ensembl→CAT (8-class) |
| B2 | `panel_B2_intron_chain_4group` | Transcript concordance Ensembl→CAT (4-group) |
| B1_cat | `panel_B1cat_intron_chain_8class` | Transcript concordance CAT→Ensembl (8-class) |
| B_compare | `panel_B_compare_directions` | Side-by-side Ens→CAT vs CAT→Ens (4-group) |
| B_ratio | `panel_B_ratio` | CAT/Ensembl transcript count ratio by biotype |
| B_supp | `panel_Bsupp_full_denom` | Full-denominator concordance (Ensembl→CAT) |
| C | `panel_C_jaccard_by_biotype` | Jaccard index distribution |
| D | `panel_D_cds_concordance` | CDS concordance (protein-coding) |
| E | `panel_E_divergence` | GRCh38 divergence categories |

Every `_data.tsv` contains exactly the numbers used to draw the plot.

In [ ]:
import os, shutil
from pathlib import Path
import matplotlib as mpl
from matplotlib import font_manager as fm

# Use local scratch for Matplotlib cache to avoid stale NFS handles
try:
    MPLDIR = Path('/tmp') / f"{os.environ.get('USER','user')}-mplconfig"
    MPLDIR.mkdir(parents=True, exist_ok=True)
    os.environ['MPLCONFIGDIR'] = str(MPLDIR)
    import matplotlib
    ttf_src = Path(matplotlib.get_data_path())/'fonts'/'ttf'
    ttf_dst = MPLDIR/'ttf'
    shutil.copytree(ttf_src, ttf_dst, dirs_exist_ok=True)
    for f in ttf_dst.glob('*.ttf'):
        fm.fontManager.addfont(str(f))
    fm._load_fontmanager(try_read_cache=False)
    mpl.rcParams.update({'svg.fonttype':'none', 'pdf.fonttype':42, 'ps.fonttype':42,
                         'font.family':'DejaVu Sans'})
except Exception as e:
    print('Font init warning:', e)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import gc

warnings.filterwarnings('ignore')

# Publication-quality defaults
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 8,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

print(f'pandas {pd.__version__}, numpy {np.__version__}')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
OUTPUT_DIR  = Path('/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results')

QC_DIR         = OUTPUT_DIR / 'qc_metrics'
RESULTS_DIR    = OUTPUT_DIR / 'results'
SUMMARY_DIR    = OUTPUT_DIR / 'summary_stats'
FIGURE_DIR     = OUTPUT_DIR / 'figures'
INTRON_DIR     = OUTPUT_DIR / 'intermediate_spreadsheets' / 'intron_chain'
CDS_DIR        = OUTPUT_DIR / 'intermediate_spreadsheets' / 'coding_integrity'
DIV_DIR        = OUTPUT_DIR / 'intermediate_spreadsheets' / 'divergence'
FIGURE_DIR.mkdir(exist_ok=True, parents=True)

def save_panel(fig, prefix):
    """Save a standalone panel as PDF + PNG."""
    fig.savefig(FIGURE_DIR / f'{prefix}.pdf', bbox_inches='tight')
    fig.savefig(FIGURE_DIR / f'{prefix}.png', dpi=300, bbox_inches='tight')
    print(f'  Saved {prefix}.{{pdf,png}}')

def save_data(df, prefix):
    """Save source data TSV alongside a panel."""
    path = FIGURE_DIR / f'{prefix}_data.tsv'
    df.to_csv(path, sep='\t', index=False)
    print(f'  Saved {prefix}_data.tsv  ({len(df)} rows)')

print(f'Output:  {OUTPUT_DIR}')
print(f'Figures: {FIGURE_DIR}')

In [ ]:
# ── Colour palette ────────────────────────────────────────────────────────

# 8-class intron chain colours (visually grouped)
CLASS_COLORS = {
    'Exact_Match':    '#2a9d8f',
    'Intron_Match':   '#264653',
    'Intron_Subset':  '#457b9d',
    'Intron_Superset':'#74a9cf',
    'Partial_5':      '#e9c46a',
    'Partial_3':      '#f4a261',
    'Other_Partial':  '#e76f51',
    'No_Match':       '#c1121f',
}
CLASS_LABELS = {
    'Exact_Match':     'Exact match',
    'Intron_Match':    'Intron chain match',
    'Intron_Subset':   'Intron subset',
    'Intron_Superset': 'Intron superset',
    'Partial_5':       "Partial (5')",
    'Partial_3':       "Partial (3')",
    'Other_Partial':   'Other partial',
    'No_Match':        'No match',
}
CLASSIFICATION_ORDER = [
    'Exact_Match', 'Intron_Match', 'Intron_Subset', 'Intron_Superset',
    'Partial_5', 'Partial_3', 'Other_Partial', 'No_Match',
]

# 4-group collapsed
GROUP_4_MAP = {
    'Exact_Match':    'Exact match',
    'Intron_Match':   'Same intron chain',
    'Intron_Subset':  'Same intron chain',
    'Intron_Superset':'Same intron chain',
    'Partial_5':      'Partial overlap',
    'Partial_3':      'Partial overlap',
    'Other_Partial':  'Partial overlap',
    'No_Match':       'No match',
}
GROUP_4_COLORS = {
    'Exact match':       '#2a9d8f',
    'Same intron chain': '#457b9d',
    'Partial overlap':   '#f4a261',
    'No match':          '#c1121f',
}
GROUP_4_ORDER = ['Exact match', 'Same intron chain', 'Partial overlap', 'No match']

BIOTYPE_ORDER  = ['protein_coding', 'lncRNA', 'pseudogene', 'other_ncRNA', 'other']
BIOTYPE_LABELS = {
    'protein_coding': 'Protein-coding',
    'lncRNA':         'lncRNA',
    'pseudogene':     'Pseudogene',
    'other_ncRNA':    'Other ncRNA',
    'other':          'Other',
}

---
## Load all data

In [ ]:
# ── Gene presence per assembly ────────────────────────────────────────────
funnel_file = SUMMARY_DIR / 'funnel_rung1_gene_presence_per_asm.tsv'
if funnel_file.exists():
    gene_pres = pd.read_csv(funnel_file, sep='\t')
    print(f'Loaded gene presence: {len(gene_pres)} assemblies')
else:
    import re
    ACC_RE = re.compile(r'(GC[AF]_\d+\.\d+)')
    files = sorted(QC_DIR.rglob('*_gene_presence.tsv'))
    print(f'Computing gene presence from {len(files)} files...')
    rows = []
    for f in files:
        m = ACC_RE.search(f.name)
        acc = m.group(1) if m else f.stem
        df = pd.read_csv(f, sep='\t')
        for col in ['present_in_ensembl', 'present_in_cat']:
            df[col] = df[col].map({'True': True, 'False': False, True: True, False: False})
        df = df[~df['gene_name'].str.match(r'^ENSG', na=False)]
        n_union = len(df)
        n_both = ((df['present_in_ensembl']) & (df['present_in_cat'])).sum()
        rows.append({'assembly_accession': acc,
                     'n_union_loci': n_union, 'n_both_loci': int(n_both),
                     'pct_gene_presence': n_both / n_union if n_union > 0 else np.nan})
    gene_pres = pd.DataFrame(rows)
    print(f'Computed gene presence: {len(gene_pres)} assemblies')

gene_pres['pct'] = gene_pres['pct_gene_presence'] * 100

# ── Intron chain classification ───────────────────────────────────────────
ic_file = INTRON_DIR / 'intron_chain_by_biotype_per_assembly.tsv'
if ic_file.exists():
    ic_data = pd.read_csv(ic_file, sep='\t')
    print(f'Loaded intron chain: {len(ic_data):,} rows, '
          f'{ic_data["assembly_accession"].nunique()} assemblies')
    if 'direction' in ic_data.columns:
        for d in ic_data['direction'].unique():
            n = len(ic_data[ic_data['direction'] == d])
            print(f'  {d}: {n:,} rows')
    else:
        # Legacy format: no direction column → assume Ensembl_to_CAT
        ic_data['direction'] = 'Ensembl_to_CAT'
        print('  (legacy format — added direction=Ensembl_to_CAT)')
else:
    print(f'WARNING: {ic_file} not found — run aggregate_intron_chain_by_biotype.py first')
    ic_data = pd.DataFrame()

# ── Transcript count ratios ───────────────────────────────────────────────
ratio_file = INTRON_DIR / 'transcript_count_ratio_per_assembly.tsv'
if ratio_file.exists():
    ratio_data = pd.read_csv(ratio_file, sep='\t')
    print(f'Loaded transcript count ratios: {len(ratio_data):,} rows')
else:
    ratio_data = pd.DataFrame()
    print(f'No transcript count ratio file found.')

# ── Jaccard by biotype ────────────────────────────────────────────────────
jac_file = INTRON_DIR / 'jaccard_by_biotype_per_assembly.tsv'
jac_data = pd.read_csv(jac_file, sep='\t') if jac_file.exists() else pd.DataFrame()
if not jac_data.empty:
    print(f'Loaded Jaccard: {len(jac_data):,} rows')

# ── CDS concordance ──────────────────────────────────────────────────────
cds_asm_file = CDS_DIR / 'coding_integrity_per_assembly.tsv'
cds_asm = pd.read_csv(cds_asm_file, sep='\t') if cds_asm_file.exists() else pd.DataFrame()
if not cds_asm.empty:
    print(f'Loaded CDS: {len(cds_asm)} assemblies')

# ── GRCh38 divergence ─────────────────────────────────────────────────────
div_asm_file = DIV_DIR / 'grch38_divergence_per_assembly.tsv'
div_asm = pd.read_csv(div_asm_file, sep='\t') if div_asm_file.exists() else pd.DataFrame()
if not div_asm.empty:
    print(f'Loaded divergence: {div_asm["assembly_accession"].nunique()} assemblies')

---
## Panel A — Gene-level agreement

Three standalone prototypes. All share the same source data TSV.

In [ ]:
# ── Source data ───────────────────────────────────────────────────────────
panel_A_data = gene_pres[['assembly_accession', 'n_union_loci', 'n_both_loci',
                           'pct_gene_presence', 'pct']].copy()
panel_A_data = panel_A_data.rename(columns={'pct': 'pct_gene_presence_x100'})
save_data(panel_A_data, 'panel_A_gene_agreement')

vals = gene_pres['pct'].dropna()
med = vals.median()
print(f'  Median: {med:.1f}%  '
      f'IQR: {vals.quantile(0.25):.1f}\u2013{vals.quantile(0.75):.1f}%  '
      f'n={len(vals)}')

In [ ]:
# ── A1: Strip + Boxplot ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(4, 5))
ax.boxplot(vals, vert=True, widths=0.5, patch_artist=True,
           boxprops=dict(facecolor='#2a9d8f', alpha=0.3),
           medianprops=dict(color='#264653', linewidth=2),
           whiskerprops=dict(color='#264653'),
           capprops=dict(color='#264653'),
           flierprops=dict(marker='o', markersize=3, alpha=0.5))
jitter = np.random.default_rng(42).uniform(-0.15, 0.15, len(vals))
ax.scatter(np.ones(len(vals)) + jitter, vals, s=6, alpha=0.35,
           color='#2a9d8f', edgecolors='none', zorder=3)
ax.set_ylabel('Gene loci detected by both methods (%)')
ax.set_xticks([1]); ax.set_xticklabels([f'n = {len(vals)} assemblies'])
ax.set_ylim(max(vals.min() - 1, 90), 100)
ax.text(1, med + 0.15, f'{med:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
save_panel(fig, 'panel_A1_strip_box')
plt.show()

In [ ]:
# ── A2: Horizontal bar + dot strip ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.barh(0, med, height=0.5, color='#2a9d8f', alpha=0.85, edgecolor='white')
ax.text(med + 0.1, 0, f'{med:.1f}%', va='center', fontsize=9, fontweight='bold')
jitter_y = np.random.default_rng(42).uniform(-0.18, 0.18, len(vals))
ax.scatter(vals, jitter_y, s=6, alpha=0.4, color='#264653', edgecolors='none', zorder=3)
ax.set_xlabel('Gene loci detected by both methods (%)')
ax.set_yticks([])
ax.set_xlim(max(vals.min() - 1, 90), 101)
ax.axvline(med, color='#264653', linewidth=0.8, linestyle='--', alpha=0.5)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
save_panel(fig, 'panel_A2_horiz_bar')
plt.show()

In [ ]:
# ── A3: Violin ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(4, 5))
parts = ax.violinplot(vals, vert=True, showmedians=True, showextrema=True)
for pc in parts['bodies']:
    pc.set_facecolor('#2a9d8f'); pc.set_alpha(0.5)
parts['cmedians'].set_color('#264653'); parts['cmedians'].set_linewidth(2)
ax.set_ylabel('Gene loci detected by both methods (%)')
ax.set_xticks([1]); ax.set_xticklabels([f'n = {len(vals)} assemblies'])
ax.set_ylim(max(vals.min() - 1, 90), 100)
ax.text(1, med + 0.15, f'{med:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
save_panel(fig, 'panel_A3_violin')
plt.show()

---
## Panel B — Transcript concordance by biotype

Two standalone figures: B1 (8-class detail) and B2 (4-group collapsed).

**Key design choice:** These panels operate at the **transcript level**, not the
gene level. Each Ensembl transcript is classified by its best CAT match, then
we count transcripts per classification. This avoids the optimistic bias of
"best-per-gene" aggregation, where a gene with 1/10 exact-matching transcripts
would look the same as a gene with 10/10.

In [ ]:
if ic_data.empty:
    print('Skipping Panel B — no intron chain data.')
else:
    # The intron chain data now has a `direction` column with two values:
    #   Ensembl_to_CAT: "can each Ensembl transcript find a CAT match?"
    #   CAT_to_Ensembl: "can each CAT transcript find an Ensembl match?"
    #
    # The Ensembl→CAT direction is typically more favorable because CAT
    # annotates more transcripts per gene, so most Ensembl transcripts
    # can find a match in the larger set. The CAT→Ensembl direction
    # reveals extra CAT transcripts that have no Ensembl counterpart.

    # Compute median + IQR per direction × biotype × classification
    medians_by_dir = {}
    stats_by_dir = {}

    for direction in ['Ensembl_to_CAT', 'CAT_to_Ensembl']:
        dir_data = ic_data[ic_data['direction'] == direction]
        if dir_data.empty:
            print(f'  No data for direction: {direction}')
            continue

        stats = (
            dir_data.groupby(['biotype', 'classification'])['pct']
            .agg(median_pct='median', q25_pct=lambda x: x.quantile(0.25),
                 q75_pct=lambda x: x.quantile(0.75), n_assemblies='count')
            .reset_index()
        )
        stats_by_dir[direction] = stats

        medians = (
            stats.pivot(index='biotype', columns='classification', values='median_pct')
            .reindex(index=BIOTYPE_ORDER, columns=CLASSIFICATION_ORDER, fill_value=0)
        )
        medians_by_dir[direction] = medians

    # Backward-compat variables for existing Panel B cells
    stats_8 = stats_by_dir.get('Ensembl_to_CAT', pd.DataFrame())
    medians_8 = medians_by_dir.get('Ensembl_to_CAT', pd.DataFrame())

    # Also compute 4-group collapsed for both directions
    medians_4_by_dir = {}
    for direction, stats in stats_by_dir.items():
        stats_4 = stats.copy()
        stats_4['group'] = stats_4['classification'].map(GROUP_4_MAP)
        stats_4_agg = (
            stats_4.groupby(['biotype', 'group'])
            .agg(median_pct=('median_pct', 'sum'),
                 q25_pct=('q25_pct', 'sum'),
                 q75_pct=('q75_pct', 'sum'))
            .reset_index()
        )
        medians_4 = (
            stats_4_agg.pivot(index='biotype', columns='group', values='median_pct')
            .reindex(index=BIOTYPE_ORDER, columns=GROUP_4_ORDER, fill_value=0)
        )
        medians_4_by_dir[direction] = medians_4

    # Backward-compat
    medians_4 = medians_4_by_dir.get('Ensembl_to_CAT', pd.DataFrame())

    # Save source data
    save_data(stats_by_dir.get('Ensembl_to_CAT', pd.DataFrame()),
              'panel_B1_intron_chain_8class')
    save_data(stats_by_dir.get('CAT_to_Ensembl', pd.DataFrame()),
              'panel_B1cat_intron_chain_8class')

    for direction, label in [('Ensembl_to_CAT', 'Ensembl → CAT'),
                              ('CAT_to_Ensembl', 'CAT → Ensembl')]:
        if direction in medians_by_dir:
            print(f'\n8-class medians — {label} (% of transcripts):')
            display(medians_by_dir[direction].round(1))

    if not ratio_data.empty:
        ratio_med = (
            ratio_data.groupby('biotype')['ratio_cat_to_ens']
            .median()
            .reindex(BIOTYPE_ORDER)
        )
        print('\nMedian CAT/Ensembl transcript count ratio per biotype:')
        display(ratio_med.round(2))

In [ ]:
if not ic_data.empty:
    # ── B1: Full 8-class (Ensembl → CAT) ──────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 4))
    y_pos = np.arange(len(BIOTYPE_ORDER))
    left = np.zeros(len(BIOTYPE_ORDER))

    for cls in CLASSIFICATION_ORDER:
        v = medians_8[cls].values
        ax.barh(y_pos, v, left=left, height=0.65,
                color=CLASS_COLORS[cls], edgecolor='white', linewidth=0.3,
                label=CLASS_LABELS[cls])
        for j, (val, l) in enumerate(zip(v, left)):
            if val >= 5:
                ax.text(l + val/2, j, f'{val:.0f}%', ha='center', va='center',
                        fontsize=7, color='white', fontweight='bold')
        left += v

    ax.set_yticks(y_pos)
    ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER])
    ax.set_xlabel('Percentage of Ensembl transcripts matched in CAT (median across assemblies)')
    ax.set_title('Ensembl \u2192 CAT', fontsize=10, fontweight='bold', loc='left')
    ax.set_xlim(0, 100)
    ax.invert_yaxis()
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    handles = [mpatches.Patch(color=CLASS_COLORS[c], label=CLASS_LABELS[c])
               for c in CLASSIFICATION_ORDER]
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.15),
              ncol=4, fontsize=7, frameon=False)

    plt.tight_layout()
    save_panel(fig, 'panel_B1_intron_chain_8class')
    plt.show()

In [ ]:
if not ic_data.empty:
    # ── B2: Collapsed 4-group (Ensembl → CAT) ─────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4))
    y_pos = np.arange(len(BIOTYPE_ORDER))
    left = np.zeros(len(BIOTYPE_ORDER))

    for grp in GROUP_4_ORDER:
        v = medians_4[grp].values
        ax.barh(y_pos, v, left=left, height=0.65,
                color=GROUP_4_COLORS[grp], edgecolor='white', linewidth=0.3,
                label=grp)
        for j, (val, l) in enumerate(zip(v, left)):
            if val >= 5:
                ax.text(l + val/2, j, f'{val:.0f}%', ha='center', va='center',
                        fontsize=7, color='white', fontweight='bold')
        left += v

    ax.set_yticks(y_pos)
    ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER])
    ax.set_xlabel('Percentage of Ensembl transcripts matched in CAT (median across assemblies)')
    ax.set_title('Ensembl \u2192 CAT', fontsize=10, fontweight='bold', loc='left')
    ax.set_xlim(0, 100)
    ax.invert_yaxis()
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    handles = [mpatches.Patch(color=GROUP_4_COLORS[g], label=g) for g in GROUP_4_ORDER]
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.15),
              ncol=4, fontsize=7, frameon=False)

    plt.tight_layout()
    save_panel(fig, 'panel_B2_intron_chain_4group')
    plt.show()

---
## Panel B_cat — CAT → Ensembl direction

This is the **reverse direction**: "what fraction of CAT transcripts can find
a match in the corresponding Ensembl gene?" Because CAT typically annotates
more transcripts per gene, the CAT→Ensembl direction reveals the 'extra'
CAT transcripts that have no Ensembl counterpart — these appear as No_Match.

In [ ]:
if not ic_data.empty and 'CAT_to_Ensembl' in medians_by_dir:
    # ── B1_cat: Full 8-class (CAT → Ensembl) ─────────────────────────────
    medians_8_cat = medians_by_dir['CAT_to_Ensembl']

    fig, ax = plt.subplots(figsize=(9, 4))
    y_pos = np.arange(len(BIOTYPE_ORDER))
    left = np.zeros(len(BIOTYPE_ORDER))

    for cls in CLASSIFICATION_ORDER:
        v = medians_8_cat[cls].values
        ax.barh(y_pos, v, left=left, height=0.65,
                color=CLASS_COLORS[cls], edgecolor='white', linewidth=0.3,
                label=CLASS_LABELS[cls])
        for j, (val, l) in enumerate(zip(v, left)):
            if val >= 5:
                ax.text(l + val/2, j, f'{val:.0f}%', ha='center', va='center',
                        fontsize=7, color='white', fontweight='bold')
        left += v

    ax.set_yticks(y_pos)
    ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER])
    ax.set_xlabel('Percentage of CAT transcripts matched in Ensembl (median across assemblies)')
    ax.set_title('CAT \u2192 Ensembl', fontsize=10, fontweight='bold', loc='left')
    ax.set_xlim(0, 100)
    ax.invert_yaxis()
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    handles = [mpatches.Patch(color=CLASS_COLORS[c], label=CLASS_LABELS[c])
               for c in CLASSIFICATION_ORDER]
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.15),
              ncol=4, fontsize=7, frameon=False)

    plt.tight_layout()
    save_panel(fig, 'panel_B1cat_intron_chain_8class')
    plt.show()
else:
    print('No CAT→Ensembl direction data available.')

In [ ]:
if not ic_data.empty and 'CAT_to_Ensembl' in medians_4_by_dir:
    # ── B_compare: Side-by-side 4-group comparison ────────────────────────
    # This is the key figure: shows that Ensembl→CAT looks ~98% exact,
    # but CAT→Ensembl reveals many extra CAT transcripts with no match.

    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

    dir_configs = [
        ('Ensembl_to_CAT', 'Ensembl \u2192 CAT',
         'Pct of Ensembl transcripts'),
        ('CAT_to_Ensembl', 'CAT \u2192 Ensembl',
         'Pct of CAT transcripts'),
    ]

    for ax, (direction, title, xlabel) in zip(axes, dir_configs):
        med4 = medians_4_by_dir[direction]
        y_pos = np.arange(len(BIOTYPE_ORDER))
        left = np.zeros(len(BIOTYPE_ORDER))

        for grp in GROUP_4_ORDER:
            v = med4[grp].values
            ax.barh(y_pos, v, left=left, height=0.65,
                    color=GROUP_4_COLORS[grp], edgecolor='white', linewidth=0.3,
                    label=grp)
            for j, (val, l) in enumerate(zip(v, left)):
                if val >= 4:
                    ax.text(l + val/2, j, f'{val:.0f}%', ha='center', va='center',
                            fontsize=7, color='white', fontweight='bold')
            left += v

        ax.set_yticks(y_pos)
        ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER])
        ax.set_xlabel(f'{xlabel} (median across assemblies)')
        ax.set_title(title, fontsize=10, fontweight='bold', loc='left')
        ax.set_xlim(0, 100)
        ax.invert_yaxis()
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    # Shared legend below
    handles = [mpatches.Patch(color=GROUP_4_COLORS[g], label=g) for g in GROUP_4_ORDER]
    fig.legend(handles=handles, loc='lower center',
               bbox_to_anchor=(0.5, -0.08), ncol=4, fontsize=7, frameon=False)

    plt.tight_layout()
    save_panel(fig, 'panel_B_compare_directions')
    plt.show()
else:
    print('Side-by-side comparison requires both directions.')

---
## Panel B_ratio — Transcript count ratio (CAT / Ensembl)

Shows how many more transcripts CAT annotates per gene compared to Ensembl.
A ratio > 1 means CAT predicts more transcripts. This explains why the
Ensembl→CAT direction shows high concordance (small set easily finds matches
in larger set) while CAT→Ensembl shows more discordance (extra CAT transcripts
have no Ensembl counterpart).

In [ ]:
if not ratio_data.empty:
    # ── B_ratio: CAT/Ensembl transcript count ratio by biotype ────────
    save_data(ratio_data, 'panel_B_ratio')

    fig, ax = plt.subplots(figsize=(8, 5))

    bio_data, bio_labels_plot, bio_colors = [], [], []
    palette = {'protein_coding': '#2a9d8f', 'lncRNA': '#264653',
               'pseudogene': '#457b9d', 'other_ncRNA': '#e9c46a',
               'other': '#f4a261'}

    for bio in BIOTYPE_ORDER:
        bio_df = ratio_data[ratio_data['biotype'] == bio]
        if bio_df.empty:
            continue
        bio_data.append(bio_df['ratio_cat_to_ens'].values)
        bio_labels_plot.append(BIOTYPE_LABELS[bio])
        bio_colors.append(palette.get(bio, '#999999'))

    positions = np.arange(1, len(bio_data) + 1)
    parts = ax.violinplot(bio_data, positions=positions, showmedians=True)
    for pc, col in zip(parts['bodies'], bio_colors):
        pc.set_facecolor(col); pc.set_alpha(0.6)

    ax.set_xticks(positions)
    ax.set_xticklabels(bio_labels_plot, rotation=25, ha='right')
    ax.set_ylabel('CAT / Ensembl transcript count ratio')
    ax.axhline(1.0, color='grey', linewidth=1, linestyle='--', alpha=0.6,
               label='Ratio = 1 (equal counts)')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    for i, d in enumerate(bio_data):
        m = np.median(d)
        ax.text(i + 1, m + 0.05, f'{m:.2f}', ha='center', fontsize=8,
                fontweight='bold')

    ax.legend(loc='upper right', fontsize=7, frameon=False)
    plt.tight_layout()
    save_panel(fig, 'panel_B_ratio')
    plt.show()
else:
    print('No transcript count ratio data available.')

---
## Panel B_supp — Full-denominator transcript concordance (supplementary)

Same as Panel B, but the denominator includes **all Ensembl transcripts**,
not just those from RBH-paired genes. Transcripts from Ensembl genes with
no CAT counterpart appear as "Gene not shared."

This shows the complete picture in one panel. Requires running both
`count_gff_transcripts.py` (on each assembly's Ensembl GFF) and
`aggregate_intron_chain_by_biotype.py --all-ensembl-genes-dir`.

In [ ]:
# ── Load full-denominator data (optional) ─────────────────────────────────
full_denom_file = INTRON_DIR / 'intron_chain_full_denom_per_assembly.tsv'
if full_denom_file.exists():
    fd_data = pd.read_csv(full_denom_file, sep='\t')
    print(f'Loaded full-denominator: {len(fd_data):,} rows, '
          f'{fd_data["assembly_accession"].nunique()} assemblies')
else:
    print(f'No full-denominator file — skipping Panel B_supp.')
    print(f'  To generate: run aggregate_intron_chain_by_biotype.py '
          f'with --all-ensembl-genes-dir')
    fd_data = pd.DataFrame()

if not fd_data.empty:
    # 5-group: same 4 groups + Gene_Not_Shared
    FULL_DENOM_GROUP_MAP = {
        'Exact_Match':     'Exact match',
        'Intron_Match':    'Same intron chain',
        'Intron_Subset':   'Same intron chain',
        'Intron_Superset': 'Same intron chain',
        'Partial_5':       'Partial overlap',
        'Partial_3':       'Partial overlap',
        'Other_Partial':   'Partial overlap',
        'No_Match':        'No match',
        'Gene_Not_Shared': 'Gene not shared',
    }
    GROUP_5_ORDER = ['Exact match', 'Same intron chain', 'Partial overlap',
                     'No match', 'Gene not shared']
    GROUP_5_COLORS = {
        'Exact match':       '#2a9d8f',
        'Same intron chain': '#457b9d',
        'Partial overlap':   '#f4a261',
        'No match':          '#c1121f',
        'Gene not shared':   '#adb5bd',
    }

    fd_data['group'] = fd_data['classification'].map(FULL_DENOM_GROUP_MAP)

    # Aggregate per assembly × biotype × group
    fd_grouped = (
        fd_data.groupby(['assembly_accession', 'biotype', 'group'])['pct']
        .sum()
        .reset_index()
    )

    # Median across assemblies
    fd_stats = (
        fd_grouped.groupby(['biotype', 'group'])['pct']
        .agg(median_pct='median', q25_pct=lambda x: x.quantile(0.25),
             q75_pct=lambda x: x.quantile(0.75))
        .reset_index()
    )
    save_data(fd_stats, 'panel_Bsupp_full_denom')

    fd_medians = (
        fd_stats.pivot(index='biotype', columns='group', values='median_pct')
        .reindex(index=BIOTYPE_ORDER, columns=GROUP_5_ORDER, fill_value=0)
    )

    print('\nFull-denominator medians (% of ALL Ensembl transcripts):')
    display(fd_medians.round(1))

    # ── Figure ────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 4))
    y_pos = np.arange(len(BIOTYPE_ORDER))
    left = np.zeros(len(BIOTYPE_ORDER))

    for grp in GROUP_5_ORDER:
        v = fd_medians[grp].values
        ax.barh(y_pos, v, left=left, height=0.65,
                color=GROUP_5_COLORS[grp], edgecolor='white', linewidth=0.3,
                label=grp)
        for j, (val, l) in enumerate(zip(v, left)):
            if val >= 4:
                ax.text(l + val/2, j, f'{val:.0f}%', ha='center', va='center',
                        fontsize=7, color='white' if grp != 'Gene not shared' else 'black',
                        fontweight='bold')
        left += v

    ax.set_yticks(y_pos)
    ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER])
    ax.set_xlabel('Percentage of ALL Ensembl transcripts (median across assemblies)')
    ax.set_xlim(0, 100)
    ax.invert_yaxis()
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    handles = [mpatches.Patch(color=GROUP_5_COLORS[g], label=g) for g in GROUP_5_ORDER]
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.15),
              ncol=5, fontsize=7, frameon=False)

    plt.tight_layout()
    save_panel(fig, 'panel_Bsupp_full_denom')
    plt.show()

---
## Panel C — Jaccard index distribution by biotype

In [ ]:
if jac_data.empty:
    print('Skipping Panel C — no Jaccard data.')
else:
    # ── Source data ────────────────────────────────────────────────────────
    save_data(jac_data, 'panel_C_jaccard_by_biotype')

    # ── Figure ────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 5))

    bio_stats, bio_labels_plot = [], []
    for bio in BIOTYPE_ORDER:
        bio_df = jac_data[jac_data['biotype'] == bio]
        if bio_df.empty:
            continue
        bio_stats.append({
            'med':    float(bio_df['median'].median()),
            'q1':     float(bio_df['p25'].median()),
            'q3':     float(bio_df['p75'].median()),
            'whislo': float(bio_df['p5'].median()),
            'whishi': float(bio_df['p95'].median()),
            'fliers': [],
        })
        bio_labels_plot.append(BIOTYPE_LABELS[bio])

    positions = np.arange(1, len(bio_stats) + 1)
    bp = ax.bxp(bio_stats, positions=positions, widths=0.55,
                patch_artist=True, showfliers=False,
                medianprops=dict(color='black', linewidth=2))
    for patch in bp['boxes']:
        patch.set_facecolor('#2a9d8f'); patch.set_alpha(0.6)

    ax.set_xticks(positions)
    ax.set_xticklabels(bio_labels_plot, rotation=25, ha='right')
    ax.set_ylabel('Jaccard index (best-match exon overlap)')
    ax.set_ylim(-0.05, 1.05)
    ax.axhline(1.0, color='grey', linewidth=0.5, linestyle=':', alpha=0.4)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    for i, stats in enumerate(bio_stats):
        ax.text(i + 1, stats['med'] + 0.03, f"{stats['med']:.2f}",
                ha='center', fontsize=8, fontweight='bold')

    plt.tight_layout()
    save_panel(fig, 'panel_C_jaccard_by_biotype')
    plt.show()

---
## Panel D — CDS concordance (protein-coding)

In [ ]:
if cds_asm.empty:
    print('Skipping Panel D — no CDS data.')
else:
    # Find the full-match column
    pct_col = None
    for candidate in ['pct_Full_Match', 'pct_full_match', 'full_match_pct']:
        if candidate in cds_asm.columns:
            pct_col = candidate
            break
    print(f'CDS columns: {list(cds_asm.columns)}')
    print(f'Using: {pct_col}')

    if pct_col:
        # ── Source data ────────────────────────────────────────────────────
        cds_export = cds_asm[['assembly_accession', pct_col]].copy()
        cds_export = cds_export.rename(columns={pct_col: 'pct_cds_full_match'})
        save_data(cds_export, 'panel_D_cds_concordance')

        cds_vals = cds_asm[pct_col].dropna()
        cds_med = cds_vals.median()

        # ── Figure ────────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(4, 5))
        ax.boxplot(cds_vals, vert=True, widths=0.5, patch_artist=True,
                   boxprops=dict(facecolor='#e76f51', alpha=0.3),
                   medianprops=dict(color='#264653', linewidth=2),
                   whiskerprops=dict(color='#264653'),
                   capprops=dict(color='#264653'),
                   flierprops=dict(marker='o', markersize=3, alpha=0.5))
        jitter = np.random.default_rng(42).uniform(-0.15, 0.15, len(cds_vals))
        ax.scatter(np.ones(len(cds_vals)) + jitter, cds_vals, s=6, alpha=0.35,
                   color='#e76f51', edgecolors='none', zorder=3)
        ax.set_ylabel('CDS Full Match (%)')
        ax.set_xticks([1]); ax.set_xticklabels([f'n = {len(cds_vals)} assemblies'])
        ax.text(1, cds_med + 0.15, f'{cds_med:.1f}%', ha='center', fontsize=9, fontweight='bold')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

        plt.tight_layout()
        save_panel(fig, 'panel_D_cds_concordance')
        plt.show()
    else:
        print(f'Could not find full-match column. Available: {list(cds_asm.columns)}')

---
## Panel E — GRCh38 divergence categories

In [ ]:
DIV_COLS   = ['pct_both_agree_reference', 'pct_both_agree_diverged',
              'pct_ensembl_specific', 'pct_cat_specific']
DIV_LABELS = ['Both agree\n(same as ref)', 'Both agree\n(diverged)',
              'Ensembl-\nspecific', 'CAT-\nspecific']
DIV_COLORS = ['#2ecc71', '#3498db', '#e67e22', '#9b59b6']

if div_asm.empty:
    print('Skipping Panel E — no divergence data.')
else:
    present_cols = [c for c in DIV_COLS if c in div_asm.columns]
    if not present_cols:
        print(f'No divergence columns found. Available: {list(div_asm.columns)[:8]}')
    else:
        # ── Source data ────────────────────────────────────────────────────
        div_export = div_asm[['assembly_accession'] + present_cols].copy()
        save_data(div_export, 'panel_E_divergence')

        # ── Figure ────────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(7, 5))
        data = [div_asm[c].dropna().values for c in present_cols]
        labels = [DIV_LABELS[DIV_COLS.index(c)] for c in present_cols]
        colors = [DIV_COLORS[DIV_COLS.index(c)] for c in present_cols]

        parts = ax.violinplot(data, showmedians=True)
        for i, (pc, col) in enumerate(zip(parts['bodies'], colors)):
            pc.set_facecolor(col); pc.set_alpha(0.6)

        ax.set_xticks(range(1, len(labels) + 1))
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_ylabel('Percentage per assembly (%)')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

        for i, d in enumerate(data):
            m = np.median(d)
            ax.text(i + 1, m + 1, f'{m:.1f}%', ha='center', fontsize=8, fontweight='bold')

        plt.tight_layout()
        save_panel(fig, 'panel_E_divergence')
        plt.show()

---
## Summary statistics for manuscript text

In [ ]:
print('=' * 60)
print('KEY NUMBERS FOR MANUSCRIPT TEXT')
print('=' * 60)

vals = gene_pres['pct'].dropna()
print(f'\nGene-level agreement:')
print(f'  Median: {vals.median():.1f}%  '
      f'(IQR: {vals.quantile(0.25):.1f}\u2013{vals.quantile(0.75):.1f}%)')
print(f'  Range:  {vals.min():.1f}\u2013{vals.max():.1f}%')
print(f'  N assemblies: {len(vals)}')

if not ic_data.empty:
    for direction, label in [('Ensembl_to_CAT', 'Ensembl \u2192 CAT'),
                              ('CAT_to_Ensembl', 'CAT \u2192 Ensembl')]:
        if direction not in medians_by_dir:
            continue
        m8 = medians_by_dir[direction]
        print(f'\nTranscript concordance \u2014 {label} (median % across assemblies):')
        for bio in BIOTYPE_ORDER:
            if bio in m8.index:
                exact = m8.loc[bio, 'Exact_Match']
                same_chain = (exact + m8.loc[bio, 'Intron_Match']
                              + m8.loc[bio, 'Intron_Subset']
                              + m8.loc[bio, 'Intron_Superset'])
                no_match = m8.loc[bio, 'No_Match']
                print(f'  {BIOTYPE_LABELS[bio]:20s}: '
                      f'Exact={exact:.1f}%, Same chain={same_chain:.1f}%, '
                      f'No match={no_match:.1f}%')

if not ratio_data.empty:
    print(f'\nTranscript count ratio (CAT / Ensembl, median across assemblies):')
    for bio in BIOTYPE_ORDER:
        bio_df = ratio_data[ratio_data['biotype'] == bio]
        if not bio_df.empty:
            r = bio_df['ratio_cat_to_ens'].median()
            n_ens = bio_df['n_ensembl_tx'].median()
            n_cat = bio_df['n_cat_tx'].median()
            print(f'  {BIOTYPE_LABELS[bio]:20s}: '
                  f'ratio={r:.2f}  '
                  f'(median {int(n_cat):,} CAT / {int(n_ens):,} Ensembl tx)')

if not jac_data.empty:
    print(f'\nJaccard index (median-of-medians across assemblies, per gene):')
    for bio in BIOTYPE_ORDER:
        bio_df = jac_data[jac_data['biotype'] == bio]
        if not bio_df.empty:
            print(f'  {BIOTYPE_LABELS[bio]:20s}: '
                  f'median={bio_df["median"].median():.3f}, '
                  f'IQR={bio_df["p25"].median():.3f}\u2013{bio_df["p75"].median():.3f}')

if not cds_asm.empty and pct_col:
    cv = cds_asm[pct_col].dropna()
    print(f'\nCDS concordance (protein-coding):')
    print(f'  Median: {cv.median():.1f}%  '
          f'(IQR: {cv.quantile(0.25):.1f}\u2013{cv.quantile(0.75):.1f}%)')

print(f'\n{"=" * 60}')
print('Output files written to:', FIGURE_DIR)
print('Done.')

---## Addendum — Denominator‑Sensitive Global Views and Rug OverlaysThis addendum adds non‑breaking panels at the end of the notebook:- Rug‑overlay variants of Panel B (4‑group), exposing per‑assembly spread while retaining median stacks.- Global, transcript‑weighted 4‑group stacks for both directions (paired‑genes denominator) — a cohort‑pooled view.- Global, transcript‑weighted 5‑group stacks for the full‑denominator Ensembl→CAT view (adds Gene not shared), when available.Each panel writes a matching `_data.tsv` with exactly the plotted values.

In [ ]:

# Helpers for rug overlays and pooled views
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Add rung dots for 4-group distributions (per-assembly) behind bars
# ic_df must have columns: assembly_accession, direction, biotype, classification, pct

def add_rung_dots_4group(ax, ic_df, direction, biotype_order, group4_map, group4_colors,
                         y_positions, group_order=None, y_offsets=None, alpha=0.18, size=6, seed=42):
    rng = np.random.default_rng(seed)
    group_order = group_order or ['Exact match','Same intron chain','Partial overlap','No match']
    y_offsets = y_offsets or {
        'Exact match':       -0.24,
        'Same intron chain': -0.08,
        'Partial overlap':    0.08,
        'No match':           0.24,
    }

    df = ic_df[ic_df['direction'] == direction].copy()
    if df.empty:
        return
    df['group'] = df['classification'].map(group4_map)

    # Collapse 8-class pct to per-assembly 4-group pct (sum within assembly×biotype)
    asm = (df.groupby(['assembly_accession','biotype','group'])['pct']
             .sum().reset_index())

    for j, bio in enumerate(biotype_order):
        bio_df = asm[asm['biotype'] == bio]
        for g in group_order:
            vals = bio_df[bio_df['group'] == g]['pct'].to_numpy()
            if vals.size == 0:
                continue
            yj = y_positions[j] + y_offsets[g] + rng.uniform(-0.03, 0.03, size=vals.size)
            ax.scatter(vals, yj, s=size, alpha=alpha, color=group4_colors[g],
                       edgecolors='none', zorder=1)

# Build tidy dataset with per-assembly 4-group pct and median lines (for _data.tsv)

def build_rug_dataset_4group(ic_df, direction, biotype_order, group4_map, group_order):
    df = ic_df[ic_df['direction'] == direction].copy()
    if df.empty:
        return pd.DataFrame()
    df['group'] = df['classification'].map(group4_map)
    asm = (df.groupby(['assembly_accession','biotype','group'])['pct']
             .sum().reset_index().rename(columns={'pct':'value'}))
    asm['kind'] = 'assembly'
    # medians per biotype×group across assemblies
    med = (asm.groupby(['biotype','group'])['value']
             .median().reset_index())
    med['assembly_accession'] = ''
    med['kind'] = 'median'
    out = pd.concat([asm[['assembly_accession','biotype','group','value','kind']],
                     med[['assembly_accession','biotype','group','value','kind']]], ignore_index=True)
    # ensure biotype order
    out['biotype'] = pd.Categorical(out['biotype'], categories=biotype_order, ordered=True)
    out['group'] = pd.Categorical(out['group'], categories=group_order, ordered=True)
    out['direction'] = direction
    return out.sort_values(['kind','biotype','group','assembly_accession'])

# Pooled 4-group (transcript-weighted) for a direction
# Uses n_transcripts to sum across assemblies, then percent within biotype

def pooled_4group(ic_df, direction, biotype_order, group4_map, group_order):
    df = ic_df[ic_df['direction'] == direction].copy()
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()
    df['group'] = df['classification'].map(group4_map)
    pooled = (df.groupby(['biotype','group'])['n_transcripts']
                .sum().reset_index(name='n'))
    totals = pooled.groupby('biotype')['n'].transform('sum')
    pooled['pct'] = 100 * pooled['n'] / totals
    wide = (pooled.pivot(index='biotype', columns='group', values='pct')
                  .reindex(index=biotype_order, columns=group_order, fill_value=0))
    wide_counts = (pooled.pivot(index='biotype', columns='group', values='n')
                          .reindex(index=biotype_order, columns=group_order, fill_value=0))
    return wide, wide_counts

# Utility: safe palette for 5-group full-denominator
FULL_DENOM_GROUP_MAP = {
    'Exact_Match':     'Exact match',
    'Intron_Match':    'Same intron chain',
    'Intron_Subset':   'Same intron chain',
    'Intron_Superset': 'Same intron chain',
    'Partial_5':       'Partial overlap',
    'Partial_3':       'Partial overlap',
    'Other_Partial':   'Partial overlap',
    'No_Match':        'No match',
    'Gene_Not_Shared': 'Gene not shared',
}
GROUP_5_ORDER = ['Exact match','Same intron chain','Partial overlap','No match','Gene not shared']
GROUP_5_COLORS = {
    'Exact match':       '#2a9d8f',
    'Same intron chain': '#457b9d',
    'Partial overlap':   '#f4a261',
    'No match':          '#c1121f',
    'Gene not shared':   '#adb5bd',
}


### Panel B2 (rug overlay) — Ensembl → CAT, 4‑group with per‑assembly rugsDenominator: transcripts from paired genes only. Bars are medians across assemblies; semi‑transparent dots are per‑assembly 4‑group percentages (collapsed from 8‑class within each assembly).

In [ ]:

if not ic_data.empty:
    direction = 'Ensembl_to_CAT'
    # Ensure medians_4 exists
    if 'medians_4_by_dir' not in globals():
        # Recompute medians_4_by_dir if needed
        stats_by_dir = {}
        for d in ['Ensembl_to_CAT','CAT_to_Ensembl']:
            dir_data = ic_data[ic_data['direction']==d]
            if dir_data.empty: 
                continue
            stats = (dir_data.groupby(['biotype','classification'])['pct']
                              .agg(median_pct='median', q25_pct=lambda x: x.quantile(0.25),
                                   q75_pct=lambda x: x.quantile(0.75))
                              .reset_index())
            stats_by_dir[d] = stats
        medians_4_by_dir = {}
        for d, stats in stats_by_dir.items():
            stats_4 = stats.copy(); stats_4['group'] = stats_4['classification'].map(GROUP_4_MAP)
            stats_4_agg = (stats_4.groupby(['biotype','group'])
                                   .agg(median_pct=('median_pct','sum'))
                                   .reset_index())
            medians_4_by_dir[d] = (stats_4_agg.pivot(index='biotype', columns='group', values='median_pct')
                                            .reindex(index=BIOTYPE_ORDER, columns=GROUP_4_ORDER, fill_value=0))
    med4 = medians_4_by_dir.get(direction)

    # Build dataset for export (assemblies + medians)
    rug_df = build_rug_dataset_4group(ic_data, direction, BIOTYPE_ORDER, GROUP_4_MAP, GROUP_4_ORDER)
    if not rug_df.empty:
        save_data(rug_df.assign(direction=direction), 'panel_B2_intron_chain_4group_rugs')

    fig, ax = plt.subplots(figsize=(8, 4))
    y_pos = np.arange(len(BIOTYPE_ORDER))

    # rugs first
    add_rung_dots_4group(ax, ic_data, direction, BIOTYPE_ORDER, GROUP_4_MAP, GROUP_4_COLORS,
                         y_pos, group_order=GROUP_4_ORDER)

    # median stacks
    left = np.zeros(len(BIOTYPE_ORDER))
    for grp in GROUP_4_ORDER:
        v = med4[grp].values if med4 is not None else np.zeros(len(BIOTYPE_ORDER))
        ax.barh(y_pos, v, left=left, height=0.65,
                color=GROUP_4_COLORS[grp], edgecolor='white', linewidth=0.3, zorder=2,
                label=grp)
        for j, (val, l) in enumerate(zip(v, left)):
            if val >= 5:
                ax.text(l + val/2, j, f'{val:.0f}%', ha='center', va='center', fontsize=7, color='white', fontweight='bold')
        left += v

    ax.set_yticks(y_pos)
    ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER])
    ax.set_xlabel('Percentage of Ensembl transcripts (median across assemblies); dots = per‑assembly')
    ax.set_xlim(0,100); ax.invert_yaxis(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    handles = [mpatches.Patch(color=GROUP_4_COLORS[g], label=g) for g in GROUP_4_ORDER]
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=4, fontsize=7, frameon=False)
    plt.tight_layout(); save_panel(fig, 'panel_B2_intron_chain_4group_rugs'); plt.show()
else:
    print('Skipping B2 rugs — no intron chain data.')


### Panel B_compare (rug overlay) — Both directions, 4‑group with per‑assembly rugsDenominators: transcripts from paired genes only. Each subplot shows median stacks and underlying per‑assembly dots.

In [ ]:

if not ic_data.empty and 'medians_4_by_dir' in globals() and 'CAT_to_Ensembl' in medians_4_by_dir:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
    configs = [
        ('Ensembl_to_CAT', 'Ensembl → CAT', 'Pct of Ensembl transcripts'),
        ('CAT_to_Ensembl', 'CAT → Ensembl', 'Pct of CAT transcripts'),
    ]

    for ax, (direction, title, xlabel) in zip(axes, configs):
        med4 = medians_4_by_dir.get(direction)
        y_pos = np.arange(len(BIOTYPE_ORDER))
        # rugs
        add_rung_dots_4group(ax, ic_data, direction, BIOTYPE_ORDER, GROUP_4_MAP, GROUP_4_COLORS,
                             y_pos, group_order=GROUP_4_ORDER)
        # bars
        left = np.zeros(len(BIOTYPE_ORDER))
        for grp in GROUP_4_ORDER:
            v = med4[grp].values if med4 is not None else np.zeros(len(BIOTYPE_ORDER))
            ax.barh(y_pos, v, left=left, height=0.65,
                    color=GROUP_4_COLORS[grp], edgecolor='white', linewidth=0.3, zorder=2)
            left += v
        ax.set_yticks(y_pos); ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER])
        ax.set_xlabel(f'{xlabel} (median across assemblies); dots = per‑assembly')
        ax.set_title(title, fontsize=10, fontweight='bold', loc='left')
        ax.set_xlim(0,100); ax.invert_yaxis(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    handles = [mpatches.Patch(color=GROUP_4_COLORS[g], label=g) for g in GROUP_4_ORDER]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, -0.08), ncol=4, fontsize=7, frameon=False)
    plt.tight_layout(); save_panel(fig, 'panel_B_compare_directions_rugs'); plt.show()
else:
    print('Skipping B_compare rugs — need both directions and medians_4_by_dir.')


### Global (pooled) 4‑group — Transcript‑weighted, paired‑genes denominatorWe pool `n_transcripts` across all assemblies per biotype×group×direction, then compute within‑biotype percentages. This avoids median‑across‑assemblies bias and reflects the cohort’s transcript composition.

In [ ]:

if not ic_data.empty:
    med4_ens, counts_ens = pooled_4group(ic_data, 'Ensembl_to_CAT', BIOTYPE_ORDER, GROUP_4_MAP, GROUP_4_ORDER)
    med4_cat, counts_cat = pooled_4group(ic_data, 'CAT_to_Ensembl', BIOTYPE_ORDER, GROUP_4_MAP, GROUP_4_ORDER)

    # Export tidy source data
    def to_tidy(med4, counts, direction):
        if med4 is None or isinstance(med4, pd.DataFrame) and med4.empty:
            return pd.DataFrame()
        m = med4.reset_index().melt(id_vars='biotype', var_name='group', value_name='pct')
        c = counts.reset_index().melt(id_vars='biotype', var_name='group', value_name='n_transcripts')
        t = m.merge(c, on=['biotype','group'], how='left')
        t['direction'] = direction
        return t
    tidy = []
    tidy.append(to_tidy(med4_ens, counts_ens, 'Ensembl_to_CAT'))
    tidy.append(to_tidy(med4_cat, counts_cat, 'CAT_to_Ensembl'))
    tidy = pd.concat([x for x in tidy if x is not None and not x.empty], ignore_index=True) if len(tidy)>0 else pd.DataFrame()
    if not tidy.empty:
        save_data(tidy, 'panel_B_global_pooled_4group')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    plots = [
        (axes[0], med4_ens, 'Ensembl → CAT', 'Pct of Ensembl transcripts (pooled)'),
        (axes[1], med4_cat, 'CAT → Ensembl', 'Pct of CAT transcripts (pooled)'),
    ]
    for ax, med4, title, xlabel in plots:
        y_pos = np.arange(len(BIOTYPE_ORDER))
        left = np.zeros(len(BIOTYPE_ORDER))
        if med4 is None or med4.empty:
            continue
        for grp in GROUP_4_ORDER:
            v = med4[grp].values
            ax.barh(y_pos, v, left=left, height=0.65,
                    color=GROUP_4_COLORS[grp], edgecolor='white', linewidth=0.3, zorder=2)
            left += v
        ax.set_yticks(y_pos); ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER])
        ax.set_xlim(0,100); ax.invert_yaxis(); ax.set_title(title, fontsize=10, fontweight='bold', loc='left')
        ax.set_xlabel(xlabel); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    handles = [mpatches.Patch(color=GROUP_4_COLORS[g], label=g) for g in GROUP_4_ORDER]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, -0.08), ncol=4, fontsize=7, frameon=False)
    plt.tight_layout(); save_panel(fig, 'panel_B_global_pooled_4group'); plt.show()
else:
    print('Skipping global pooled 4-group — no intron chain data.')


In [ ]:

if not ic_data.empty:
    def cohort_pooled_4group(ic_df, direction):
        df = ic_df[ic_df['direction']==direction].copy()
        if df.empty:
            return pd.Series(dtype=float)
        df['group'] = df['classification'].map(GROUP_4_MAP)
        pooled = df.groupby(['group'])['n_transcripts'].sum()
        total = pooled.sum()
        return 100 * pooled / total if total>0 else pooled

    s_ens = cohort_pooled_4group(ic_data, 'Ensembl_to_CAT')
    s_cat = cohort_pooled_4group(ic_data, 'CAT_to_Ensembl')

    tidy = (pd.concat([
                s_ens.rename('pct').reset_index().assign(direction='Ensembl_to_CAT'),
                s_cat.rename('pct').reset_index().assign(direction='CAT_to_Ensembl')
            ], ignore_index=True)
            if (s_ens is not None and not s_ens.empty) or (s_cat is not None and not s_cat.empty)
            else pd.DataFrame())
    if not tidy.empty:
        save_data(tidy, 'panel_B_global_pooled_4group_cohort')

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
    for ax, series, title in [
        (axes[0], s_ens, 'Ensembl → CAT (cohort pooled)'),
        (axes[1], s_cat, 'CAT → Ensembl (cohort pooled)'),
    ]:
        if series is None or series.empty:
            continue
        left = 0.0
        for g in GROUP_4_ORDER:
            val = float(series.get(g, 0.0))
            ax.barh([0], [val], left=[left], height=0.5,
                    color=GROUP_4_COLORS[g], edgecolor='white', linewidth=0.3)
            if val >= 5:
                ax.text(left + val/2, 0, f'{val:.0f}%', ha='center', va='center', fontsize=8, color='white', fontweight='bold')
            left += val
        ax.set_yticks([]); ax.set_xlim(0,100); ax.set_xlabel('Pct of transcripts (pooled)')
        ax.set_title(title, fontsize=10, fontweight='bold', loc='left')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    handles = [mpatches.Patch(color=GROUP_4_COLORS[g], label=g) for g in GROUP_4_ORDER]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, -0.2), ncol=4, fontsize=7, frameon=False)
    plt.tight_layout(); save_panel(fig, 'panel_B_global_pooled_4group_cohort'); plt.show()
else:
    print('Skipping cohort pooled 4-group — no intron chain data.')


### Global (pooled) 5‑group — Transcript‑weighted, full denominator (Ensembl → CAT)Includes transcripts from Ensembl genes with no CAT pair as “Gene not shared.” This is the conservative coverage view; only available if `intron_chain_full_denom_per_assembly.tsv` was generated.

In [ ]:

# Load full-denominator data if not already loaded in earlier cell
full_denom_file = INTRON_DIR / 'intron_chain_full_denom_per_assembly.tsv'
if full_denom_file.exists():
    fd_data2 = pd.read_csv(full_denom_file, sep='	')
    print(f'Loaded full-denominator (global pooled): {len(fd_data2):,} rows, {fd_data2["assembly_accession"].nunique()} assemblies')
else:
    fd_data2 = pd.DataFrame()
    print('No full-denominator file found — skipping global pooled 5-group.')

if not fd_data2.empty:
    df = fd_data2.copy()
    df['group'] = df['classification'].map(FULL_DENOM_GROUP_MAP)
    # Pooled counts across assemblies per biotype×group
    pooled = (df.groupby(['biotype','group'])['n_transcripts']
                .sum().reset_index(name='n'))
    totals = pooled.groupby('biotype')['n'].transform('sum')
    pooled['pct'] = 100 * pooled['n'] / totals
    wide = (pooled.pivot(index='biotype', columns='group', values='pct')
                  .reindex(index=BIOTYPE_ORDER, columns=GROUP_5_ORDER, fill_value=0))

    # Export tidy source data
    tidy = pooled.rename(columns={'n':'n_transcripts'})
    save_data(tidy, 'panel_B_global_pooled_5group_full_denom')

    fig, ax = plt.subplots(figsize=(10, 4))
    y_pos = np.arange(len(BIOTYPE_ORDER))
    left = np.zeros(len(BIOTYPE_ORDER))
    for grp in GROUP_5_ORDER:
        v = wide[grp].values
        ax.barh(y_pos, v, left=left, height=0.65,
                color=GROUP_5_COLORS[grp], edgecolor='white', linewidth=0.3)
        left += v
    ax.set_yticks(y_pos); ax.set_yticklabels([BIOTYPE_LABELS[b] for b in BIOTYPE_ORDER])
    ax.set_xlim(0,100); ax.invert_yaxis(); ax.set_xlabel('Pct of Ensembl transcripts (pooled, full denominator)')
    ax.set_title('Ensembl → CAT (full denominator, pooled)', fontsize=10, fontweight='bold', loc='left')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    handles = [mpatches.Patch(color=GROUP_5_COLORS[g], label=g) for g in GROUP_5_ORDER]
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=5, fontsize=7, frameon=False)
    plt.tight_layout(); save_panel(fig, 'panel_B_global_pooled_5group_full_denom'); plt.show()


In [ ]:

if not fd_data2.empty:
    s = (fd_data2.assign(group=lambda x: x['classification'].map(FULL_DENOM_GROUP_MAP))
                     .groupby('group')['n_transcripts'].sum())
    total = s.sum()
    pct = 100 * s / total if total>0 else s
    tidy = pct.rename('pct').reset_index()
    save_data(tidy, 'panel_B_global_pooled_5group_full_denom_cohort')

    fig, ax = plt.subplots(figsize=(6.5, 2.8))
    left = 0.0
    for g in GROUP_5_ORDER:
        val = float(pct.get(g, 0.0))
        ax.barh([0], [val], left=[left], height=0.5,
                color=GROUP_5_COLORS[g], edgecolor='white', linewidth=0.3)
        if val >= 5:
            ax.text(left + val/2, 0, f'{val:.0f}%', ha='center', va='center', fontsize=8, color='white' if g!='Gene not shared' else 'black', fontweight='bold')
        left += val
    ax.set_yticks([]); ax.set_xlim(0,100); ax.set_xlabel('Pct of Ensembl transcripts (pooled, full denominator)')
    ax.set_title('Ensembl → CAT (cohort pooled, full denominator)', fontsize=10, fontweight='bold', loc='left')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    handles = [mpatches.Patch(color=GROUP_5_COLORS[g], label=g) for g in GROUP_5_ORDER]
    ax.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, -0.35), ncol=5, fontsize=7, frameon=False)
    plt.tight_layout(); save_panel(fig, 'panel_B_global_pooled_5group_full_denom_cohort'); plt.show()
